In [0]:
%sql
CREATE OR REPLACE TABLE email_triage.gold.gld_fact_ticket_sla AS
SELECT 
    ticket_id,
    CAST(first_response_time AS DATE) AS response_date,
    HOUR(first_response_time) AS response_hour,
    ticket_status,
    ticket_channel,
    ai_category,
    ai_sentiment,
    ai_urgency,
    assigned_pic,
    routing_status,
    confidence_score,
    csat_rating,
    CASE 
        WHEN ai_urgency = 'High' THEN 4.0
        WHEN ai_urgency = 'Medium' THEN 24.0
        ELSE 48.0 
    END AS sla_target_hours,
    
    ROUND(
        ABS(CAST(UNIX_TIMESTAMP(time_to_resolution) - UNIX_TIMESTAMP(first_response_time) AS DOUBLE)) / 3600.0, 
        2
    ) AS actual_resolution_hours,
    
    CASE 
        WHEN time_to_resolution IS NULL THEN NULL
        WHEN ABS(UNIX_TIMESTAMP(time_to_resolution) - UNIX_TIMESTAMP(first_response_time)) / 3600.0 > 
             (CASE WHEN ai_urgency = 'High' THEN 4.0 WHEN ai_urgency = 'Medium' THEN 24.0 ELSE 48.0 END)
        THEN 1 
        ELSE 0 
    END AS is_sla_breached
FROM email_triage.silver.slv_tickets;

In [0]:
%sql
CREATE OR REPLACE TABLE email_triage.gold.gld_dim_pic AS
SELECT * FROM VALUES
  ('Manual_Review_Queue', 'Dispatcher Team', 'Unclassified', 100),
  ('Senior_Escalation_Team', 'Senior Management', 'All Critical Issues', 50),
  ('Finance_Team', 'Billing & Finance', 'Billing & Payment', 300),
  ('IT_Support_Team', 'IT & Engineering', 'Technical Issue', 300),
  ('Logistics_Team', 'Supply Chain & Ops', 'Order & Logistics', 400),
  ('General_CS_Team', 'Customer Service', 'Product Inquiry', 500)
AS t(assigned_pic, department, specialty, max_monthly_capacity);

In [0]:
%sql
CREATE OR REPLACE VIEW email_triage.gold.fact_tickets_sla_denorm AS
SELECT 
    f.*,
    d.department,
    d.specialty
FROM email_triage.gold.gld_fact_ticket_sla f
LEFT JOIN email_triage.gold.gld_dim_pic d
    ON f.assigned_pic = d.assigned_pic;

In [0]:
%sql
SELECT 
    assigned_pic,
    COUNT(ticket_id) AS total_tickets,
    SUM(CASE WHEN is_sla_breached = 1 THEN 1 ELSE 0 END) AS breached_tickets,
    ROUND(SUM(CASE WHEN is_sla_breached = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(ticket_id), 2) AS breach_rate_percentage,
    ROUND(AVG(actual_resolution_hours), 2) AS avg_resolution_hours
FROM email_triage.gold.gld_fact_ticket_sla
WHERE ticket_status = 'Closed'
GROUP BY assigned_pic
ORDER BY breach_rate_percentage DESC;